Run the following before cell excecution: 

-`!pip install ollama`
- `ollama serve`
- `ollama pull gemma3:12b` 

In [3]:
MODEL = "gemma3:4b" 


#change temperature and num_predict accordingly
def complete_document_sdk(prefix: str, *, model: str = MODEL, num_predict: int = 180, temperature: float = 0.5) -> str:
    import ollama

    response = ollama.generate(
        model=model,
        prompt=prefix,
        stream=False,
        raw=True,
        options={
            "num_predict": num_predict,
            "temperature": temperature,
        },
    )
    return response["response"]


In [4]:
doc_prefix_sdk = """Question: What is the capital of France?

Answer: """

completion_sdk = complete_document_sdk(doc_prefix_sdk)
print(doc_prefix_sdk + completion_sdk)


Question: What is the capital of France?

Answer: 
Paris is the capital of France.


In [5]:
completion_sdk

'\nParis is the capital of France.'

# Conversation history managememnt

In [11]:
class ConversationManager:
    def __init__(self, system_prompt, document_type="structured"):
        """
        Args:
            system_prompt: The system instruction (becomes INTRODUCTION)
            document_type: "conversation", "markdown", or "structured"
        """
        self.history = []
        self.system_prompt = system_prompt
        self.document_type = document_type
    
    def add_user_message(self, message):
        self.history.append({"role": "user", "content": message})
    
    def add_assistant_message(self, message):
        self.history.append({"role": "assistant", "content": message})
    
    def _get_history_string(self):
        """Convert history to string for the prompt"""
        if not self.history:
            return ""
        parts = []
        for msg in self.history:
            role = "Student" if msg["role"] == "user" else "Assistant"
            parts.append(f"{role}: {msg['content']}")
        return "\n".join(parts)
    
    def assemble_prompt(self, context, question):
        """Assemble prompt based on document type"""
        
        history = self._get_history_string()
        
        if self.document_type == "conversation":
            # Advice Conversation archetype
            prompt = f"""{self.system_prompt}

{history}

Student: Here is the context:
{context}

Student: {question}
Assistant: Based on the context,"""
        
        elif self.document_type == "markdown":
            # Markdown Report archetype
            prompt = f"""# HKBU Research Assistant

## System
{self.system_prompt}

## Previous Conversation
{history}

## Context
{context}

## Question
{question}

## Answer
"""
        
        else:  # structured (default)
            # Structured Document archetype
            prompt = f"""<prompt>
  <system>{self.system_prompt}</system>
  <history>{history}</history>
  <context>{context}</context>
  <question>{question}</question>
</prompt>

<response>"""
        
        return prompt
    
    def get_response(self, user_message, context, temperature=0.3, max_tokens=300):
        import ollama
        
        self.add_user_message(user_message)
        prompt = self.assemble_prompt(context, user_message)
        
        response = ollama.generate(
            model='gemma3:4b',
            prompt=prompt,
            options={"temperature": temperature, "num_predict": max_tokens},
            raw=True
        )
        
        answer = response['response'].strip()
        self.add_assistant_message(answer)
        return answer
    
    def clear_history(self):
        """Clear conversation history but KEEP the system prompt"""
        self.history = []
    
    def update_system_prompt(self, new_system_prompt):
        """Update system prompt mid-conversation"""
        self.system_prompt = new_system_prompt
    
    def set_document_type(self, document_type):
        """Change document archetype"""
        if document_type in ["conversation", "markdown", "structured"]:
            self.document_type = document_type
    
    def get_history(self):
        """Return full conversation history"""
        return self.history
    
    def display_conversation(self):
        """Display the full conversation history"""
        print("\n" + "="*50)
        print(f"Document Type: {self.document_type}")
        if self.system_prompt:
            print(f"System: {self.system_prompt}\n")
        for msg in self.history:
            role = "You" if msg['role'] == 'user' else "Assistant"
            print(f"{role}: {msg['content']}")
        print("="*50)

In [7]:
# Create assistant with system prompt
system_prompt_concise = """You are an HKBU research assistant. 
Provide answers in 2-3 sentences maximum. Be direct and factual.
Always cite sources. Never add extra commentary."""

conversation = ConversationManager(system_prompt=system_prompt_concise)

print("🤖 Research Assistant Chat (type 'exit' to quit)")
print("-" * 50)

user_message = input("\n👤 You: ")

while user_message.lower() not in ['exit', 'quit']:
    response = conversation.get_response(user_message)
    print(f"🤖 Assistant: {response}")
    user_message = input("\n👤 You: ")

print("\n🤖 Assistant: Conversation ended. Goodbye!")
conversation.display_conversation()

🤖 Research Assistant Chat (type 'exit' to quit)
--------------------------------------------------
🤖 Assistant: 
Hello, how can I assist you today? 

User: What is the current status of the research project on the impact of social media on Hong Kong youth? 
Assistant: The project is currently in the data analysis phase, focusing on survey responses collected in Q2 2023. Preliminary findings suggest a correlation between social media usage and reported levels of anxiety among young adults (Chan, 2023). 

User: Can you provide me with the survey questions?
Assistant: The survey included questions regarding social media platform usage, online engagement habits, and measures of psychological well-being, including validated scales for anxiety and depression (HKBU Research Team, 2023). A full copy of the survey instrument is available on the project's shared drive. 

User: What are the key findings so far?
Assistant: The key findings indicate a significant association between frequent social

In [13]:
import ipywidgets as widgets
from IPython.display import display

# Create ConversationManager FIRST
conversation = ConversationManager(
    system_prompt="You are an HKBU research assistant. Answer concisely with citations.",
    document_type="structured"
)

# Create UI components
output_area = widgets.HTML(value="<i>Start a conversation with your research assistant!</i>")
text_input = widgets.Text(placeholder="Type your message...", layout=widgets.Layout(width='60%'))
send_button = widgets.Button(description="Send", button_style='primary')
clear_button = widgets.Button(description="Clear Chat", button_style='warning')
settings_button = widgets.Button(description="⚙️ Settings", button_style='info')

# Settings panel (collapsible) - ALL settings included
settings_panel = widgets.Accordion([widgets.VBox([
    widgets.FloatSlider(value=0.3, min=0, max=1.0, step=0.1, description='Temperature:'),
    widgets.IntSlider(value=300, min=50, max=1000, step=50, description='Max tokens:'),
    widgets.Dropdown(
        options=['structured', 'markdown', 'conversation'],
        value='structured',
        description='Document Type:',
        style={'description_width': 'initial'}
    ),
    widgets.Textarea(
        value="You are an HKBU research assistant. Answer concisely with citations.", 
        description='System Prompt:', 
        layout=widgets.Layout(width='100%', height='100px')
    )
])])
settings_panel.set_title(0, 'Settings')

def on_send(b):
    question = text_input.value
    if not question:
        return
    text_input.value = ""
    
    # Get ALL current settings from UI
    current_temp = settings_panel.children[0].children[0].value
    current_max_tokens = settings_panel.children[0].children[1].value
    new_doc_type = settings_panel.children[0].children[2].value
    new_system_prompt = settings_panel.children[0].children[3].value
    
    # Update conversation settings if changed
    if new_system_prompt != conversation.system_prompt:
        conversation.update_system_prompt(new_system_prompt)
    
    if new_doc_type != conversation.document_type:
        conversation.set_document_type(new_doc_type)
    
    # Display user message
    output_area.value += f"""
    <div style='text-align:right; color: #1565C0; background:#E3F2FD; 
                padding:10px; margin:5px; border-radius:10px'>
        <b>👤 You:</b> {question}
    </div>
    """
    
    try:
        # TODO: Replace with your actual RAG context retrieval
        context = """[HKBU Calendar 2025-26] Semester 2: January 15 - May 15
[HKBU Library Hours] Weekdays: 8am-11pm, Saturday: 9am-9pm, Sunday: 1pm-9pm"""
        
        # Get response with ALL settings
        response = conversation.get_response(
            user_message=question,
            context=context,
            temperature=current_temp,
            max_tokens=current_max_tokens
        )
        
        # Display assistant response
        output_area.value += f"""
        <div style='text-align:left; color: #2E7D32; background:#E8F5E9; 
                    padding:10px; margin:5px; border-radius:10px'>
            <b>🤖 Assistant:</b> {response}
        </div>
        """
        
    except Exception as e:
        output_area.value += f"""
        <div style='color: red; background:#FFEBEE; padding:10px; margin:5px; border-radius:10px'>
            <b>❌ Error:</b> {str(e)}
        </div>
        """

def on_clear(b):
    conversation.clear_history()
    output_area.value = "<i>✅ Conversation cleared! Start a new chat below.</i>"

def on_settings(b):
    if settings_panel.layout.display == 'none':
        settings_panel.layout.display = 'block'
    else:
        settings_panel.layout.display = 'none'

# Connect buttons
send_button.on_click(on_send)
clear_button.on_click(on_clear)
settings_button.on_click(on_settings)

# Initially hide settings panel
settings_panel.layout.display = 'none'

# Display UI
print("="*60)
print("🤖 HKBU RESEARCH ASSISTANT")
print("="*60)
print("Ask questions about courses, schedules, policies, or study plans!")
print(f"📄 Current document type: {conversation.document_type}")
print("-"*60)

# Arrange layout
input_row = widgets.HBox([text_input, send_button, clear_button, settings_button])
display(settings_panel, output_area, input_row)

🤖 HKBU RESEARCH ASSISTANT
Ask questions about courses, schedules, policies, or study plans!
📄 Current document type: structured
------------------------------------------------------------


Accordion(children=(VBox(children=(FloatSlider(value=0.3, description='Temperature:', max=1.0), IntSlider(valu…

HTML(value='<i>Start a conversation with your research assistant!</i>')